# Notebook 10: second classifier (SVM) on the five medical tabular datasets

This notebook asks whether the main findings depend on the 5-nearest-neighbour classifier used inside the fitness of Notebook 8. It repeats the tabular benchmark with an **RBF support vector machine**. SVM is a margin-based classifier, unlike distance-based KNN.

**Everything else is identical to Notebook 8:**
- Datasets: WDBC, Hepatitis, Dermatology, SPECTF and Backache, from the same PMLB files with the same checksums.
- Splits: the same stratified 70/30 split per seed, seeds 0–9. Min-max scaling is fitted on the training part.
- Fitness: 0.99 × (1 − 5-fold CV balanced accuracy) + 0.01 × (fraction of features kept).
- Budget and start: population 20, budget 1,020 evaluations, dense start.
- Test: balanced accuracy and macro-F1 of the same classifier retrained on the training part.

**Classifier:** `SVC(kernel="rbf", C=1.0, gamma="scale")`, i.e. scikit-learn's defaults, fixed before the runs and not tuned.

**Methods:**
- RL-DOHO, DO, HO, DOHO, GA, BPSO and BGWO.
- GA+SI and HO+SI.
- RL-DOHO with PERTURB only.
- LASSO, mRMR, ReliefF and all features.

**Runtime:** one SVM fitness evaluation takes about 20 ms, so expect about 1–1.5 hours. To run in parallel, set `DATASETS` to part of the list in one Colab session and the rest in another; each dataset saves to its own file. Every run is saved as soon as it finishes, so rerunning resumes where it stopped.

### Hypotheses (fixed before any run of this notebook; all outcomes are reported)

**H9: the main findings of Notebook 8 replicate with an SVM fitness.** Each part is tested with a pooled two-sided sign test over the 50 paired runs:
- **H9a** RL-DOHO reaches lower fitness than DO.
- **H9b** RL-DOHO against HO, no direction predicted.
- **H9c** the GA against RL-DOHO, no direction predicted.
- **H9d** HO+SI reaches lower fitness than HO; GA+SI against the GA, no direction predicted.
- **H9e** RL-DOHO against PERTURB only, no direction predicted.

All other comparisons are reported with the same per-dataset Friedman and Holm-corrected Wilcoxon tests and pooled sign tests as Notebook 8. A difference below 1e-12 counts as a tie.

In [1]:
# ================= 1. SETUP =================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os, time, pickle, warnings, urllib.request, math, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, precision_score, recall_score
warnings.filterwarnings("ignore")

Mounted at /content/drive


In [2]:
# ================= 2. CONFIG =================
ROOT     = '/content/drive/MyDrive' if IN_COLAB else '.'
OUT_DIR  = f"{ROOT}/rl_doho_results_v4_svm"              # results of this notebook
KNN_DIR  = f"{ROOT}/rl_doho_results_v3_knn"              # Notebook 8 (KNN results, for the comparison at the end)
DATA_DIR = f"{OUT_DIR}/data"
TAB      = {"WDBC": "breast_cancer_wisconsin_diagnostic", "Hepatitis": "hepatitis", "Dermatology": "dermatology",
            "SPECTF": "spectf", "Backache": "backache"}
DATASETS = list(TAB)                 # to run in parallel: e.g. ["WDBC", "Hepatitis"] in one session, the rest in another
SEEDS    = list(range(10))
POP_SIZE, BUDGET = 20, 20 * 51       # 1,020 evaluations, as in Notebook 8
CV_FOLDS, TEST_SIZE = 5, 0.3
ALPHA, BETA = 0.99, 0.01
SCORE    = "balanced_accuracy"
def make_clf(): return SVC(kernel="rbf", C=1.0, gamma="scale")
VERBOSE  = False

FAST = False                         # True: 1 dataset x 1 seed with a tiny budget, only to check the notebook runs
if FAST: DATASETS, SEEDS, BUDGET = ["Hepatitis"], [0], 120

os.makedirs(DATA_DIR, exist_ok=True)
FAMILY   = ["RL-DOHO", "DO", "HO", "DOHO"]
WRAPPERS = FAMILY + ["GA", "BPSO", "BGWO"]
ADDONS   = ["GA+SI", "HO+SI"]
FILTERS  = ["LASSO", "mRMR", "ReliefF", "All features"]
METHODS  = WRAPPERS + FILTERS
ALL      = WRAPPERS + ADDONS + ["RL-DOHO [PERTURB only]"] + FILTERS

def save_pkl(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)
def load_pkl(path):
    for p in (path, path + ".tmp"):
        if os.path.exists(p):
            try:
                with open(p, "rb") as f: return pickle.load(f)
            except Exception as e: print(f"WARNING: could not read {p} ({e})")
    return {}
print("Output folder:", OUT_DIR, "| datasets:", DATASETS)

Output folder: /content/drive/MyDrive/rl_doho_results_v4_svm | datasets: ['WDBC', 'Hepatitis', 'Dermatology', 'SPECTF', 'Backache']


## 3. Data (the same PMLB files as Notebook 8)

In [3]:
PMLB = "https://media.githubusercontent.com/media/EpistasisLab/pmlb/master/datasets/{0}/{0}.tsv.gz"
SHA = {"breast_cancer_wisconsin_diagnostic": "b3845ddf3afff808cd6449daa546caf4d8b18f109b7ebb2344990e4c37fea291",
       "hepatitis": "87df1bbfb83b8204093ccd84ed18bb3695220c03aa4096f63a84cc5e136dbc0b",
       "dermatology": "c8852ba546e92249debecd9a897cda3f94fd10872fa13c62eb14ab40e279166e",
       "spectf": "664081130aa48fc189b2e1d3c44332f0b3eb1ee6d31b79af1643c137697677e4",
       "backache": "21d506c397dfeb3edbbbc253b923f59be6edb516689677ebef535296c6c62242"}
DATA = {}
for short in DATASETS:
    name = TAB[short]
    path = next((p for p in [f"{KNN_DIR}/data/{name}.tsv.gz", f"{DATA_DIR}/{name}.tsv.gz"] if os.path.exists(p)), f"{DATA_DIR}/{name}.tsv.gz")
    if not os.path.exists(path): urllib.request.urlretrieve(PMLB.format(name), path)
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()
    if sha != SHA[name]: print(f"WARNING: {name} checksum differs from Notebook 8's file")
    df = pd.read_csv(path, sep="\t")
    DATA[short] = (df.drop(columns="target").values.astype(float), pd.factorize(df["target"], sort=True)[0])
pd.DataFrame([{"dataset": n, "samples": X.shape[0], "features": X.shape[1], "classes": len(np.unique(y))} for n, (X, y) in DATA.items()])

,dataset,samples,features,classes
0,WDBC,569,30,2
1,Hepatitis,155,19,2
2,Dermatology,366,34,6
3,SPECTF,349,44,2
4,Backache,180,32,2


## 4. Fitness and test evaluation (SVM; otherwise identical to Notebook 8)

In [4]:
def mask_str(m): return np.packbits(np.asarray(m, dtype=np.uint8)).tobytes().hex()
NEW_EVAL = {}
def use_dataset(name, seed):
    global X_tr, X_te, y_tr, y_te, N_FEATURES, _cache, CV, CUR
    X, y = DATA[name]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    sc = MinMaxScaler().fit(X_tr); X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
    N_FEATURES = X_tr.shape[1]; CV = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=seed); CUR = (name, seed)
    _cache = load_pkl(f"{OUT_DIR}/cache_{name}_s{seed}.pkl")
def save_cache_now(): save_pkl(_cache, f"{OUT_DIR}/cache_{CUR[0]}_s{CUR[1]}.pkl")

def fitness_function(mask):
    key = mask_str(mask)
    if key in _cache: return _cache[key]["fit"]
    sel = np.flatnonzero(mask)
    if len(sel) == 0:
        _cache[key] = {"fit": 1.0, "acc": 0.0}; return 1.0
    t0 = time.perf_counter()
    acc = cross_val_score(make_clf(), X_tr[:, sel], y_tr, cv=CV, scoring=SCORE).mean()
    f = ALPHA * (1 - acc) + BETA * len(sel) / len(mask)
    _cache[key] = {"fit": f, "acc": acc}
    c = NEW_EVAL.setdefault(CUR[0], [0, 0.0]); c[0] += 1; c[1] += time.perf_counter() - t0
    return f

def test_metrics(mask):
    sel = np.flatnonzero(mask)
    p = make_clf().fit(X_tr[:, sel], y_tr).predict(X_te[:, sel])
    return {"test_acc": float(balanced_accuracy_score(y_te, p)), "test_prec": float(precision_score(y_te, p, average="macro", zero_division=0)),
            "test_rec": float(recall_score(y_te, p, average="macro", zero_division=0)), "test_f1": float(f1_score(y_te, p, average="macro"))}
print("Fitness ready.")

Fitness ready.


## 5. Algorithms (the Notebook 8 code, unchanged)

In [5]:
# ===================== RL-DOHO v2: shared algorithm code =====================
# Every method gets the same number of FITNESS EVALUATIONS (not iterations), because the
# published HO evaluates about 3N candidates per iteration while DO evaluates N.
# Encoding: continuous position x in [LB, UB]^D, feature j selected if sigmoid(x_j) > 0.5 (i.e. x_j > 0).
import math, time
import numpy as np, pandas as pd

LB, UB = -3.0, 3.0

class BudgetExhausted(Exception):
    pass

def binarize(x):
    return (np.asarray(x) > 0).astype(np.int8)          # identical to sigmoid(x) > 0.5

def mkey(mask):
    return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()

class Problem:
    """Wraps a (cached, deterministic) fitness function and enforces the evaluation budget.
    Also records the time spent inside the fitness function (fsec) and the number of distinct
    subsets evaluated (len(seen)), so the algorithm's own overhead can be separated from fitness cost."""
    def __init__(self, fitness_fn, D, budget):
        self.fitness_fn, self.D, self.budget, self.calls = fitness_fn, D, budget, 0
        self.fsec, self.seen = 0.0, set()
    def left(self):
        return self.budget - self.calls
    def f(self, mask):
        if self.calls >= self.budget:
            raise BudgetExhausted
        self.calls += 1
        m = np.asarray(mask, dtype=np.int8); self.seen.add(mkey(m))
        t0 = time.perf_counter(); v = float(self.fitness_fn(m)); self.fsec += time.perf_counter() - t0
        return v
    def fx(self, x):
        return self.f(binarize(x))

# ------------------------------------------------------------------ initialisation
def init_population(rng, N, D, mode="dense"):
    """dense : x ~ U(-1, 1)  -> each feature selected with probability 0.5 (as in Notebooks 1-5).
       sparse: agent i keeps a fraction p_i ~ log-uniform[min(10/D, 0.1), 0.1] of the features
               (used when D > 100: ten features up to 10% of them)."""
    if mode == "dense":
        return rng.uniform(-1, 1, (N, D))
    lo, hi = math.log(min(10.0 / D, 0.1)), math.log(0.1)
    X = np.empty((N, D))
    for i in range(N):
        p = math.exp(rng.uniform(lo, hi))
        sel = rng.random(D) < p
        if not sel.any():
            sel[rng.integers(D)] = True
        X[i] = np.where(sel, rng.uniform(0.1, 1.0, D), -rng.uniform(0.1, 1.0, D))
    return X

def random_position(rng, D, mode):
    return init_population(rng, 1, D, mode)[0]

# ------------------------------------------------------------------ Levy flight (Mantegna)
def levy(rng, shape, beta=1.5):
    num = math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
    den = math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2)
    sigma = (num / den) ** (1 / beta)
    u = rng.normal(0, sigma, shape); v = rng.normal(0, 1, shape)
    return u / np.abs(v) ** (1 / beta)

def lognpdf(x):                                          # lognormal pdf, mu = 0, sigma = 1
    x = np.maximum(x, 1e-300)
    return np.exp(-np.log(x) ** 2 / 2) / (x * math.sqrt(2 * math.pi))

# ------------------------------------------------------------------ Dandelion Optimizer (Zhao et al., 2022)
def do_move(rng, X, elite, t, T):
    """One full DO iteration: rising (Eqs. 2-8), descending (Eqs. 10-11), landing (Eqs. 12-14).
    Returns the new population; the caller evaluates it (N evaluations)."""
    N, D = X.shape
    T = max(T, 2); t = min(max(t, 1), T)
    alpha = rng.random() * ((t / T) ** 2 - 2 * t / T + 1)                     # Eq. 4: adaptive step
    a = 1.0 / (T ** 2 - 2 * T + 1); b = -2 * a; c = 1 - a - b
    k = 1 - rng.random() * (c + a * t ** 2 + b * t)                          # Eq. 8 (k with q)
    # --- rising stage
    if rng.standard_normal() < 1.5:                                          # clear weather
        lamb = np.abs(rng.standard_normal((N, D)))
        theta = (2 * rng.random(N) - 1) * math.pi
        r = 1 / np.exp(theta); vx = r * np.cos(theta); vy = r * np.sin(theta)
        Xs = rng.uniform(LB, UB, (N, D))                                     # Eq. 3: random position
        X1 = X + alpha * (vx * vy)[:, None] * lognpdf(lamb) * (Xs - X)       # Eq. 2
    else:                                                                    # rainy weather
        X1 = X * k                                                           # Eq. 7
    X1 = np.clip(X1, LB, UB)
    # --- descending stage (Brownian motion around the population mean)
    beta = rng.standard_normal((N, D))
    Xm = X1.mean(axis=0)                                                     # Eq. 11
    X2 = np.clip(X1 - beta * alpha * (Xm - beta * alpha * X1), LB, UB)      # Eq. 10
    # --- landing stage (Levy flight around the elite)
    step = 0.01 * levy(rng, (N, D))                                          # Eq. 13, s = 0.01, beta = 1.5
    delta = 2 * t / T                                                        # Eq. 14
    X3 = elite + step * alpha * (elite - X2 * delta)                         # Eq. 12
    return np.clip(X3, LB, UB)

# ------------------------------------------------------------------ Hippopotamus Optimization (Amiri et al., 2024)
def ho_iteration(rng, X, F, dominant, t, T, prob, on_eval):
    """One full HO iteration with greedy selection, updating X, F in place.
    Phase 1 (first half): male + female/immature candidates (Eqs. 3-9)      -> 2 evaluations per agent
    Phase 2 (second half): predator + defence candidate (Eqs. 10-16)        -> 2 evaluations per agent
    Phase 3 (all agents): escape within shrinking local bounds (Eqs. 17-19) -> 1 evaluation per agent
    on_eval(x, f) is called for every evaluated candidate that is a hippopotamus position."""
    N, D = X.shape
    half = N // 2
    T = max(T, 1); t = min(max(t, 1), T)
    for i in range(half):
        I1, I2 = rng.integers(1, 3, 2); rho = rng.integers(0, 2, 2)
        g = int(rng.integers(1, N + 1)); grp = rng.choice(N, g, replace=False)
        MG = X[grp].mean(axis=0)
        h = [I2 * rng.random(D) + (1 - rho[0]), 2 * rng.random(D) - 1, rng.random(D),
             I1 * rng.random(D) + (1 - rho[1]), rng.random()]                  # Eq. 4
        h1, h2 = h[rng.integers(5)], h[rng.integers(5)]
        x_m = np.clip(X[i] + rng.random() * (dominant - I1 * X[i]), LB, UB)     # Eq. 3 (male)
        if math.exp(-t / T) > 0.6:                                            # Eq. 5
            x_f = X[i] + h1 * (dominant - I2 * MG)                            # Eq. 6
        elif rng.random() > 0.5:
            x_f = X[i] + h2 * (MG - dominant)                                 # Eq. 7
        else:
            x_f = rng.uniform(LB, UB, D)
        x_f = np.clip(x_f, LB, UB)
        for x in (x_m, x_f):                                                  # Eqs. 8-9: greedy
            f = prob.fx(x); on_eval(x, f)
            if f < F[i]: X[i], F[i] = x, f
    for i in range(half, N):
        pred = rng.uniform(LB, UB, D)                                         # Eq. 10
        f_pred = prob.fx(pred)
        dist = np.abs(pred - X[i]) + 1e-12                                    # Eq. 11
        b_, c_, d_ = rng.uniform(2, 4), rng.uniform(1, 1.5), rng.uniform(2, 3)
        l_ = rng.uniform(-2 * math.pi, 2 * math.pi)                           # 2*pi*g with g in [-1, 1]
        RL = 0.05 * levy(rng, D)                                              # Eq. 13
        if F[i] > f_pred:
            x = RL * pred + (b_ / (c_ - d_ * math.cos(l_))) * (1 / dist)      # Eq. 12, case 1
        else:
            x = RL * pred + (b_ / (c_ - d_ * math.cos(l_))) * (1 / (2 * dist + rng.random(D)))
        x = np.clip(x, LB, UB)
        f = prob.fx(x); on_eval(x, f)
        if f < F[i]: X[i], F[i] = x, f                                        # Eqs. 15-16
    lo, hi = LB / t, UB / t                                                   # Eq. 17: local bounds
    for i in range(N):
        s = [2 * rng.random(D) - 1, rng.random(), rng.standard_normal()][rng.integers(3)]   # Eq. 19
        x = np.clip(X[i] + rng.random() * (lo + s * (hi - lo)), LB, UB)      # Eq. 18
        f = prob.fx(x); on_eval(x, f)
        if f < F[i]: X[i], F[i] = x, f                                        # Eq. 20

# ------------------------------------------------------------------ run bookkeeping
class Tracker:
    """Tracks the best subset found and logs one row per iteration."""
    def __init__(self, name, prob, verbose=False, tie_eps=0.0):
        self.name, self.prob, self.verbose, self.tie_eps = name, prob, verbose, tie_eps
        self.best_f, self.best_x, self.best_m = np.inf, None, None
        self.rows, self.t0 = [], time.time()
    def better(self, f, nf):
        if self.best_m is None: return True
        bf, bn = self.best_f, int(self.best_m.sum())
        e = self.tie_eps
        if e <= 0: return f < bf
        return f < bf - e or (abs(f - bf) <= e and (nf < bn or (nf == bn and f < bf)))
    def offer(self, x, f, mask=None):
        m = binarize(x) if mask is None else mask
        if self.better(f, int(m.sum())):
            self.best_f, self.best_x, self.best_m = f, (None if x is None else np.array(x, float)), m.copy()
            return True
        return False
    def log(self, it, action, F=None, reward=None):
        row = dict(iter=it, evals=self.prob.calls, action=action, best=self.best_f,
                   n_feats=int(self.best_m.sum()), pop_mean=(float(np.mean(F)) if F is not None else np.nan),
                   reward=reward, sec=time.time() - self.t0)
        self.rows.append(row)
        if self.verbose:
            rw = "" if reward is None else f"  r={reward:.3f}"
            print(f"  {self.name:<16} it {it:>3}  evals {self.prob.calls:>5}/{self.prob.budget}  {action:<10}"
                  f" best {self.best_f:.5f}  feats {row['n_feats']:>5}{rw}")
    def result(self):
        return {"mask": self.best_m, "fit": self.best_f, "hist": pd.DataFrame(self.rows)}

def _start(prob, rng, N, init, tr):
    X = init_population(rng, N, prob.D, init)
    F = np.array([prob.fx(x) for x in X])
    for x, f in zip(X, F): tr.offer(x, f)
    tr.log(0, "init", F)
    return X, F

# ------------------------------------------------------------------ DO, HO, DOHO (published update rules)
def run_do(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("DO", prob, verbose)
    T = max(2, (prob.budget - N) // N)
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            X = do_move(rng, X, tr.best_x, t, T)
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            tr.log(t, "DO", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_ho(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("HO", prob, verbose)
    T = max(1, math.ceil((prob.budget - N) / (3 * N)))
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            dom = X[np.argmin(F)].copy()
            ho_iteration(rng, X, F, dom, t, T, prob, tr.offer)
            tr.log(t, "HO", F)
    except BudgetExhausted:
        if tr.rows and tr.rows[-1]["evals"] != prob.calls: tr.log(len(tr.rows), "HO (partial)", F)
    return tr.result()

def run_doho(prob, N, seed, init="dense", verbose=False):
    """Fixed switch: published DO for the first half of the budget, then published HO."""
    rng = np.random.default_rng(seed); tr = Tracker("DOHO", prob, verbose)
    half_budget = prob.budget // 2
    T1 = max(2, (half_budget - N) // N); T2 = max(1, math.ceil((prob.budget - half_budget) / (3 * N)))
    it = 0
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T1 + 1):
            if prob.calls + N > half_budget: break
            X = do_move(rng, X, tr.best_x, t, T1)
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            it += 1; tr.log(it, "DO", F)
        for t in range(1, T2 + 1):
            ho_iteration(rng, X, F, X[np.argmin(F)].copy(), t, T2, prob, tr.offer)
            it += 1; tr.log(it, "HO", F)
    except BudgetExhausted:
        pass
    return tr.result()

# ------------------------------------------------------------------ controllers
class UCB1:
    def __init__(self, n_arms, seed, c=1.0):
        self.rng = np.random.default_rng(seed + 7919); self.c = c
        self.n = np.zeros(n_arms); self.v = np.zeros(n_arms)
    def choose(self, state=None):
        untried = np.flatnonzero(self.n == 0)
        if len(untried): return int(self.rng.choice(untried)), "i"
        ucb = self.v + self.c * np.sqrt(2 * np.log(self.n.sum()) / self.n)
        return int(self.rng.choice(np.flatnonzero(np.isclose(ucb, ucb.max())))), "u"
    def update(self, a, r):
        self.n[a] += 1; self.v[a] += (r - self.v[a]) / self.n[a]
    def finish(self):
        pass

class RandomArm(UCB1):
    def choose(self, state=None): return int(self.rng.integers(len(self.n))), "r"

class FixedArm(UCB1):
    def __init__(self, n_arms, seed, arm=0):
        super().__init__(n_arms, seed); self.arm = arm
    def choose(self, state=None): return self.arm, "f"

N_STATES = 6
def controller_state(calls, budget, last_gain):
    """State for the Q-learning controller: budget phase (first, second, last third) x whether the
    previous intervention improved the best subset (0/1) -> 6 states."""
    phase = min(2, int(3 * calls / budget))
    return 2 * phase + (1 if last_gain > 0 else 0)

class QLearn(UCB1):
    """Tabular one-step Q-learning over the arms. Epsilon-greedy (eps = 0.2), learning rate 0.3,
    discount 0.5, Q initialised at 0. The next state is the state at the next decision; the last
    decision of a run is updated without bootstrapping. Settings fixed in advance, not tuned."""
    def __init__(self, n_arms, seed, n_states=N_STATES, lr=0.3, gamma=0.5, eps=0.2):
        super().__init__(n_arms, seed)
        self.Q = np.zeros((n_states, n_arms)); self.lr, self.gamma, self.eps = lr, gamma, eps
        self.pending, self.s = None, 0
    def choose(self, state=0):
        if self.pending is not None:
            s, a, r = self.pending
            self.Q[s, a] += self.lr * (r + self.gamma * self.Q[state].max() - self.Q[s, a]); self.pending = None
        self.s = state
        if self.rng.random() < self.eps:
            return int(self.rng.integers(self.Q.shape[1])), "e"
        q = self.Q[state]
        return int(self.rng.choice(np.flatnonzero(np.isclose(q, q.max())))), "q"
    def update(self, a, r):
        self.n[a] += 1; self.pending = (self.s, a, r)
    def finish(self):
        if self.pending is not None:
            s, a, r = self.pending; self.Q[s, a] += self.lr * (r - self.Q[s, a]); self.pending = None

CONTROLLERS = {"ucb": UCB1, "random": RandomArm, "qlearn": QLearn}

ARMS = ("PERTURB", "RESTART", "DO", "HO")

# ------------------------------------------------------------------ RL-DOHO v2
def run_rldoho(prob, N, seed, init="dense", verbose=False, controller="ucb", reward="gain",
               kappa=2, tie_eps=1e-3, k_max=3, tabu_tries=20, intervene=True, name="RL-DOHO"):
    """DO (published) while the best improves; after `kappa` iterations without an accepted improvement,
    a controller picks one arm: PERTURB / RESTART (worse half, tabu) or one published DO / HO iteration.
    reward="gain": 0.8 * min(1, g / g_max) + 0.2 * share, where g = fitness gain of the best per evaluation
                   spent (parsimony-only moves give g = 0) and g_max = largest g seen so far in the run.
    reward="old" : 1 if the best changed, otherwise 0.2 * share (Notebooks 1-5)."""
    rng = np.random.default_rng(seed); D = prob.D
    tr = Tracker(name, prob, verbose, tie_eps=tie_eps)
    ctrl = CONTROLLERS.get(controller)
    ctrl = ctrl(len(ARMS), seed) if ctrl else FixedArm(len(ARMS), seed, arm=ARMS.index(controller))
    T_do = max(2, (prob.budget - N) // N); T_ho = max(1, math.ceil((prob.budget - N) / (3 * N)))
    def sched(T):                     # DO / HO schedules follow the share of the budget already spent
        return min(T, max(1, math.ceil(T * prob.calls / prob.budget)))
    tabu = set(); g_max = 0.0; stag = 0; it = 0; last_gain = 0.0
    arm_log = []
    def offer(x, f):
        tabu.add(mkey(binarize(x)))
        return tr.offer(x, f)
    def tabu_draw(make):
        for _ in range(tabu_tries):
            x = np.clip(make(), LB, UB)
            if mkey(binarize(x)) not in tabu: break
        return x
    try:
        X = init_population(rng, N, D, init)
        F = np.array([prob.fx(x) for x in X])
        for x, f in zip(X, F): offer(x, f)
        tr.log(0, "init", F)
        while prob.left() > 0:
            it += 1
            prev_best, prev_feats, prev_med, calls0 = tr.best_f, int(tr.best_m.sum()), float(np.median(F)), prob.calls
            changed_before = (tr.best_f, int(tr.best_m.sum()))
            if not intervene or stag < kappa:
                arm, how = None, None
                X = do_move(rng, X, tr.best_x, sched(T_do), T_do)
                F = np.array([prob.fx(x) for x in X])
                for x, f in zip(X, F): offer(x, f)
                moved = np.arange(N)
            else:
                arm, how = ctrl.choose(controller_state(prob.calls, prob.budget, last_gain)); a = ARMS[arm]
                if a == "DO":
                    X = do_move(rng, X, tr.best_x, sched(T_do), T_do)
                    F = np.array([prob.fx(x) for x in X])
                    for x, f in zip(X, F): offer(x, f)
                    moved = np.arange(N)
                elif a == "HO":
                    F_before = F.copy()
                    ho_iteration(rng, X, F, X[np.argmin(F)].copy(), sched(T_ho), T_ho, prob, offer)
                    moved = np.flatnonzero(F != F_before)
                else:
                    worst = np.argsort(F)[N // 2:]
                    for i in worst:
                        if a == "PERTURB":
                            def make():
                                x = tr.best_x.copy(); k = int(rng.integers(1, k_max + 1))
                                j = rng.choice(D, k, replace=False)
                                x[j] = -np.sign(x[j] + 1e-12) * rng.uniform(0.5, 2.0, k)
                                return x
                        else:
                            make = lambda: random_position(rng, D, init)
                        X[i] = tabu_draw(make)
                        F[i] = prob.fx(X[i]); offer(X[i], F[i])
                    moved = worst
            cost = max(1, prob.calls - calls0)
            changed = (tr.best_f, int(tr.best_m.sum())) != changed_before
            stag = 0 if changed else stag + 1
            g = max(0.0, prev_best - tr.best_f) / cost
            g_max = max(g_max, g)
            if arm is None:
                tr.log(it, "explore", F)
            else:
                share = float(np.mean(F[moved] < prev_med)) if len(moved) else 0.0
                if reward == "gain":
                    r = 0.8 * (min(1.0, g / g_max) if g_max > 0 else 0.0) + 0.2 * share
                else:
                    r = 1.0 if changed else 0.2 * share
                ctrl.update(arm, r); last_gain = prev_best - tr.best_f
                arm_log.append((it, ARMS[arm], how, r, prev_best - tr.best_f, cost))
                tr.log(it, f"{ARMS[arm]}({how})", F, reward=r)
    except BudgetExhausted:
        pass
    ctrl.finish()
    res = tr.result()
    res["arms"] = pd.DataFrame(arm_log, columns=["iter", "arm", "how", "reward", "gain", "cost"])
    return res

# ------------------------------------------------------------------ GA, BPSO, BGWO (same budget and initial population)
def run_ga(prob, N, seed, init="dense", verbose=False, pc=0.9, tour=3):
    rng = np.random.default_rng(seed); tr = Tracker("GA", prob, verbose); D = prob.D; pm = 1.0 / D
    try:
        X, F = _start(prob, rng, N, init, tr)
        M = binarize(X); it = 0
        while True:
            def pick():
                idx = rng.choice(N, tour, replace=False); return M[idx[np.argmin(F[idx])]]
            new = [tr.best_m.copy()]
            while len(new) < N:
                p1, p2 = pick(), pick()
                if rng.random() < pc:
                    cm = rng.random(D) < 0.5; kids = [np.where(cm, p1, p2), np.where(cm, p2, p1)]
                else:
                    kids = [p1.copy(), p2.copy()]
                for c in kids:
                    c = np.where(rng.random(D) < pm, 1 - c, c).astype(np.int8)
                    if len(new) < N: new.append(c)
            M = np.array(new, dtype=np.int8)
            F = np.empty(N)
            for i in range(N):
                F[i] = prob.f(M[i]); tr.offer(None, F[i], mask=M[i])
            it += 1; tr.log(it, "gen", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_bpso(prob, N, seed, init="dense", verbose=False, c1=2.0, c2=2.0, vmax=6.0):
    rng = np.random.default_rng(seed); tr = Tracker("BPSO", prob, verbose); D = prob.D
    T = max(2, (prob.budget - N) // N)
    try:
        X0, F = _start(prob, rng, N, init, tr)
        X = binarize(X0); V = rng.uniform(-1, 1, (N, D))
        if init == "sparse":                      # start velocities consistent with the sparse masks
            V = np.where(X == 1, rng.uniform(0, 1, (N, D)), rng.uniform(-6, -1, (N, D)))
        pb, pbf = X.copy(), F.copy()
        for t in range(1, T + 1):
            w = 0.9 - 0.5 * t / T
            r1, r2 = rng.random((N, D)), rng.random((N, D))
            V = np.clip(w * V + c1 * r1 * (pb - X) + c2 * r2 * (tr.best_m - X), -vmax, vmax)
            X = (rng.random((N, D)) < 1 / (1 + np.exp(-V))).astype(np.int8)
            F = np.empty(N)
            for i in range(N):
                F[i] = prob.f(X[i]); tr.offer(None, F[i], mask=X[i])
            imp = F < pbf; pb[imp] = X[imp]; pbf[imp] = F[imp]
            tr.log(t, "move", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_bgwo(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("BGWO", prob, verbose); D = prob.D
    T = max(2, (prob.budget - N) // N)
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            a = 2 - 2 * t / T
            leaders = X[np.argsort(F)[:3]].copy()
            new = np.empty_like(X)
            for i in range(N):
                xs = [Lp - (2 * a * rng.random(D) - a) * np.abs(2 * rng.random(D) * Lp - X[i]) for Lp in leaders]
                new[i] = np.clip(np.mean(xs, axis=0), LB, UB)
            X = new
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            tr.log(t, "move", F)
    except BudgetExhausted:
        pass
    return tr.result()


# ------------------------------------------------------------------ stagnation-triggered add-on for any base optimiser
def ga_generation(rng, X, F, best_m, prob, offer, pc=0.9, tour=3):
    """One GA generation (tournament 3, uniform crossover, bit-flip 1/D, elitism of 1) on the masks of X.
    Children are stored as positions +-1 so that the shared PERTURB / RESTART operators can act on them."""
    N, D = X.shape; M = binarize(X); pm = 1.0 / D
    def pick():
        idx = rng.choice(N, tour, replace=False); return M[idx[np.argmin(F[idx])]]
    new = [best_m.copy()]
    while len(new) < N:
        p1, p2 = pick(), pick()
        if rng.random() < pc:
            cm = rng.random(D) < 0.5; kids = [np.where(cm, p1, p2), np.where(cm, p2, p1)]
        else:
            kids = [p1.copy(), p2.copy()]
        for c in kids:
            c = np.where(rng.random(D) < pm, 1 - c, c).astype(np.int8)
            if len(new) < N: new.append(c)
    X = np.where(np.array(new) == 1, 1.0, -1.0)
    F = np.empty(N)
    for i in range(N):
        F[i] = prob.fx(X[i]); offer(X[i], F[i])
    return X, F

def run_si(prob, N, seed, base="GA", init="dense", verbose=False, controller="ucb", reward="gain",
           kappa=2, tie_eps=1e-3, k_max=3, tabu_tries=20, name=None):
    """The RL-DOHO intervention layer around another optimiser ("GA" or "HO"): the base optimiser runs while
    the best subset improves; after `kappa` stalled iterations the controller picks PERTURB, RESTART or one
    base iteration, with the same gain reward, tabu memory and tie-break as RL-DOHO. Used to test whether the
    interventions help other optimisers too."""
    rng = np.random.default_rng(seed); D = prob.D; arms = ("PERTURB", "RESTART", base)
    tr = Tracker(name or f"{base}+SI", prob, verbose, tie_eps=tie_eps)
    ctrl = CONTROLLERS[controller](len(arms), seed)
    T_ho = max(1, math.ceil((prob.budget - N) / (3 * N)))
    sched = lambda T: min(T, max(1, math.ceil(T * prob.calls / prob.budget)))
    tabu = set(); g_max = 0.0; stag = 0; it = 0; last_gain = 0.0; arm_log = []
    def offer(x, f):
        tabu.add(mkey(binarize(x))); return tr.offer(x, f)
    def base_step(X, F):
        if base == "GA":
            X, F = ga_generation(rng, X, F, tr.best_m, prob, offer); return X, F, np.arange(N)
        F0 = F.copy(); ho_iteration(rng, X, F, X[np.argmin(F)].copy(), sched(T_ho), T_ho, prob, offer)
        return X, F, np.flatnonzero(F != F0)
    try:
        X = init_population(rng, N, D, init)
        F = np.array([prob.fx(x) for x in X])
        for x, f in zip(X, F): offer(x, f)
        tr.log(0, "init", F)
        while prob.left() > 0:
            it += 1
            prev_best, prev_med, calls0 = tr.best_f, float(np.median(F)), prob.calls
            before = (tr.best_f, int(tr.best_m.sum()))
            if stag < kappa:
                arm = None; X, F, moved = base_step(X, F)
            else:
                arm, how = ctrl.choose(controller_state(prob.calls, prob.budget, last_gain)); a = arms[arm]
                if a == base:
                    X, F, moved = base_step(X, F)
                else:
                    worst = np.argsort(F)[N // 2:]
                    for i in worst:
                        if a == "PERTURB":
                            def make():
                                x = tr.best_x.copy(); k = int(rng.integers(1, k_max + 1))
                                j = rng.choice(D, k, replace=False)
                                x[j] = -np.sign(x[j] + 1e-12) * rng.uniform(0.5, 2.0, k)
                                return x
                        else:
                            make = lambda: random_position(rng, D, init)
                        for _ in range(tabu_tries):
                            x = np.clip(make(), LB, UB)
                            if mkey(binarize(x)) not in tabu: break
                        X[i] = x; F[i] = prob.fx(x); offer(x, F[i])
                    moved = worst
            cost = max(1, prob.calls - calls0)
            changed = (tr.best_f, int(tr.best_m.sum())) != before
            stag = 0 if changed else stag + 1
            g = max(0.0, prev_best - tr.best_f) / cost; g_max = max(g_max, g)
            if arm is None:
                tr.log(it, "explore", F)
            else:
                share = float(np.mean(F[moved] < prev_med)) if len(moved) else 0.0
                r = (0.8 * (min(1.0, g / g_max) if g_max > 0 else 0.0) + 0.2 * share) if reward == "gain" \
                    else (1.0 if changed else 0.2 * share)
                ctrl.update(arm, r); last_gain = prev_best - tr.best_f
                arm_log.append((it, arms[arm], how, r, prev_best - tr.best_f, cost))
                tr.log(it, f"{arms[arm]}({how})", F, reward=r)
    except BudgetExhausted:
        pass
    ctrl.finish()
    res = tr.result(); res["arms"] = pd.DataFrame(arm_log, columns=["iter", "arm", "how", "reward", "gain", "cost"])
    return res

def run_ga_si(prob, N, seed, init="dense", verbose=False, **kw): return run_si(prob, N, seed, "GA", init, verbose, **kw)
def run_ho_si(prob, N, seed, init="dense", verbose=False, **kw): return run_si(prob, N, seed, "HO", init, verbose, **kw)

# ------------------------------------------------------------------ filter / embedded baselines (as in Notebooks 4 and 5)
# Each ranks features on the training data only; k is then chosen by the same fitness (CV on training data).
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif

def rank_lasso(X, y, seed, k_max):
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-12)
    m = LogisticRegression(penalty="l1", solver="saga", C=0.5, max_iter=3000, random_state=seed).fit(Xs, y)
    return np.argsort(-np.abs(m.coef_).sum(0), kind="stable")

def rank_mrmr(X, y, seed, k_max):
    rel = mutual_info_classif(X, y, random_state=seed)
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-12); n = len(Xs)
    sel = [int(np.argmax(rel))]; red = np.zeros(X.shape[1])
    for _ in range(min(k_max, X.shape[1], 2000) - 1):
        red += np.abs(Xs.T @ Xs[:, sel[-1]]) / n
        score = rel - red / len(sel); score[sel] = -np.inf
        sel.append(int(np.argmax(score)))
    rest = [j for j in np.argsort(-rel) if j not in set(sel)]
    return np.array(sel + rest)

def rank_relieff(X, y, seed, k_max, m=500, k=10):
    rng = np.random.default_rng(seed)
    Xn = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-12)
    classes, counts = np.unique(y, return_counts=True); prior = dict(zip(classes, counts / len(y)))
    idx = rng.choice(len(Xn), min(m, len(Xn)), replace=False); W = np.zeros(Xn.shape[1])
    for i in idx:
        d = np.abs(Xn - Xn[i]).sum(1); d[i] = np.inf
        for c in classes:
            cand = np.flatnonzero(y == c); cand = cand[cand != i]
            if len(cand) == 0: continue
            nn = cand[np.argsort(d[cand])[:k]]
            diff = np.abs(Xn[nn] - Xn[i]).mean(0)
            if c == y[i]: W -= diff / len(idx)
            else: W += prior[c] / (1 - prior[y[i]]) * diff / len(idx)
    return np.argsort(-W, kind="stable")

RANKERS = {"LASSO": rank_lasso, "mRMR": rank_mrmr, "ReliefF": rank_relieff}

def k_grid(D):
    if D <= 40: return list(range(1, D + 1))
    return sorted({k for k in [5, 10, 20, 50, 100, 200, 500, 1000, 2000, D] if k <= D})

def run_filter(name, seed, X_rank, y_rank, fitness_fn, D):
    """Returns (mask, fitness, chosen k). 'All features' keeps everything."""
    if name == "All features":
        m = np.ones(D, np.int8); return m, fitness_fn(m), D
    ks = k_grid(D); order = RANKERS[name](np.asarray(X_rank, float), np.asarray(y_rank), seed, max(ks))
    best = None
    for k in ks:
        m = np.zeros(D, np.int8); m[order[:k]] = 1
        f = fitness_fn(m)
        if best is None or f < best[1] - 1e-12: best = (m, f, k)
    return best

RUNNERS = {"DO": run_do, "HO": run_ho, "DOHO": run_doho, "RL-DOHO": run_rldoho,
           "GA": run_ga, "BPSO": run_bpso, "BGWO": run_bgwo, "GA+SI": run_ga_si, "HO+SI": run_ho_si}

print("Algorithms ready:", list(RUNNERS))

Algorithms ready: ['DO', 'HO', 'DOHO', 'RL-DOHO', 'GA', 'BPSO', 'BGWO', 'GA+SI', 'HO+SI']


In [6]:
# ================= Statistics helpers =================
from scipy.stats import wilcoxon, friedmanchisquare, binomtest
TIE = 1e-12

def holm(ps):
    ps = np.asarray(ps, float); order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for i, j in enumerate(order):
        run = max(run, (m - i) * ps[j]); adj[j] = min(1.0, run)
    return adj

def wtl(a, b, higher_better):
    d = np.asarray(a) - np.asarray(b)
    w = int(np.sum(d > TIE) if higher_better else np.sum(d < -TIE)); t = int(np.sum(np.abs(d) <= TIE))
    return w, t, len(d) - w - t

def per_dataset_tests(R, ref, methods, metric, higher_better):
    """Friedman over `methods` (seeds as blocks) and Holm-corrected Wilcoxon of `ref` vs each other method."""
    rows = []
    for ds in R.dataset.unique():
        P = R[R.dataset == ds].pivot_table(index="seed", columns="method", values=metric)[methods].dropna()
        try: fp = friedmanchisquare(*[P[c] for c in methods]).pvalue
        except ValueError: fp = np.nan
        tmp = []
        for c in methods:
            if c == ref: continue
            w, t, l = wtl(P[ref], P[c], higher_better)
            try: p = wilcoxon(P[ref], P[c], zero_method="zsplit").pvalue
            except ValueError: p = 1.0
            tmp.append(dict(dataset=ds, metric=metric, vs=c, W=w, T=t, L=l, mean_diff=float((P[ref] - P[c]).mean()), p=p, friedman_p=fp))
        for r_, h in zip(tmp, holm([x["p"] for x in tmp])): r_["holm_p"] = h; rows.append(r_)
    return pd.DataFrame(rows)

def pooled_sign(R, ref, others, metric, higher_better):
    rows = []
    A = R[R.method == ref].set_index(["dataset", "seed"])[metric]
    for c in others:
        B = R[R.method == c].set_index(["dataset", "seed"])[metric]
        a, b = A.align(B, join="inner")
        w, t, l = wtl(a, b, higher_better)
        p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
        rows.append(dict(metric=metric, vs=c, W=w, T=t, L=l, n=len(a), sign_p=p))
    out = pd.DataFrame(rows); out["holm_p"] = holm(out.sign_p.values) if len(out) else []
    return out

def mean_ranks(R, methods, metric, higher_better):
    X = R[R.method.isin(methods)].copy()
    X["r"] = X.groupby(["dataset", "seed"])[metric].rank(ascending=not higher_better)
    return X.pivot_table(index="method", columns="dataset", values="r").reindex(methods)

def summary_table(R, methods):
    g = R[R.method.isin(methods)].groupby(["dataset", "method"])
    S = g.agg(fit=("fit", "mean"), fit_sd=("fit", "std"), cv=("cv", "mean"), test_acc=("test_acc", "mean"),
              acc_sd=("test_acc", "std"), test_f1=("test_f1", "mean"), n_feats=("n_feats", "mean")).reset_index()
    return S
print("Statistics helpers ready.")

def ablation_report(A, main, variants, label):
    """A: DataFrame dataset, variant, seed, fit, ... ; compares `main` with every other variant."""
    A = A.copy(); A["r"] = A.groupby(["dataset", "seed"]).fit.rank()
    print(f"\n=== {label}: mean per-run rank on fitness (1 = best) ===")
    print(A.pivot_table(index="variant", columns="dataset", values="r").reindex(variants).round(2).assign(
        ALL=A.groupby("variant").r.mean().reindex(variants).round(2)).to_string())
    print(f"\nMean fitness / test accuracy / features:")
    print(A.groupby("variant")[["fit", "test_acc", "n_feats"]].mean().reindex(variants).round(4).to_string())
    P = A.pivot_table(index=["dataset", "seed"], columns="variant", values="fit")
    print(f"\n{main} against each variant (fitness, all runs pooled): W/T/L for {main}, two-sided sign test")
    for v in variants:
        if v == main or v not in P: continue
        a, b = P[main].dropna().align(P[v].dropna(), join="inner")
        w, t, l = wtl(a, b, False); p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
        print(f"  vs {v:<28s} {w:>3}/{t}/{l:<3}  p = {p:.4f}")

# ================= Extra helpers for the extension notebooks =================
def record(res, prob, t0, test_fn, cache, key_fn):
    m = np.asarray(res["mask"]).astype(int); sec = time.time() - t0
    return {"fit": float(res["fit"]), "cv": float(cache[key_fn(m)]["acc"]), "n_feats": int(m.sum()), "mask": np.packbits(m),
            "n": len(m), "hist": res["hist"], "arms": res.get("arms"), "calls": prob.calls, "sec": sec,
            "fsec": prob.fsec, "overhead_sec": sec - prob.fsec, "n_unique": len(prob.seen), **test_fn(m)}

def frame(store, cols=("fit", "cv", "n_feats", "test_acc", "test_prec", "test_rec", "test_f1")):
    return pd.DataFrame([{"dataset": k[0], "method": k[1], "seed": k[2], **{c: v.get(c, np.nan) for c in cols}} for k, v in store.items()])

def sign_line(R, a, b, label, metric="fit", higher_better=False):
    P = R.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
    if a not in P or b not in P: print(f"{label}: missing runs"); return None
    x, y = P[a].dropna().align(P[b].dropna(), join="inner")
    w, t, l = wtl(x, y, higher_better); p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
    print(f"{label}: {a} W/T/L = {w}/{t}/{l} over {len(x)} paired runs (mean diff {float((x - y).mean()):+.5f}), sign test p = {p:.4f}")
    return dict(label=label, a=a, b=b, W=w, T=t, L=l, n=len(x), p=p)
print("Extension helpers ready.")

Statistics helpers ready.
Extension helpers ready.


## 6. All runs (resumable; one results file per dataset)

In [7]:
def res_file(ds): return f"{OUT_DIR}/v4_runs_{ds}.pkl"
def load_runs():
    out = {}
    for f in sorted(os.listdir(OUT_DIR)):
        if f.startswith("v4_runs_") and f.endswith(".pkl"): out.update(load_pkl(f"{OUT_DIR}/{f}"))
    return out
runs = load_runs(); print(f"{len(runs)} runs already saved")
JOBS = [(a, RUNNERS[a], {}) for a in WRAPPERS + ADDONS] + [("RL-DOHO [PERTURB only]", run_rldoho, dict(controller="PERTURB"))]
t_start = time.time()
for ds in DATASETS:
    for s in SEEDS:
        todo = [j for j in JOBS if (ds, j[0], s) not in runs]; todo_f = [a for a in FILTERS if (ds, a, s) not in runs]
        if not (todo or todo_f): continue
        use_dataset(ds, s)
        for name, fn, kw in todo:
            t0 = time.time(); prob = Problem(fitness_function, N_FEATURES, BUDGET)
            runs[(ds, name, s)] = record(fn(prob, POP_SIZE, s, init="dense", verbose=VERBOSE, **kw), prob, t0, test_metrics, _cache, mask_str)
            save_pkl({k: v for k, v in runs.items() if k[0] == ds}, res_file(ds))
        for a in todo_f:
            t0 = time.time(); m, f, k = run_filter(a, s, X_tr, y_tr, fitness_function, N_FEATURES)
            runs[(ds, a, s)] = {"fit": float(f), "cv": float(_cache[mask_str(m)]["acc"]), "n_feats": int(np.sum(m)), "k": k,
                                "sec": time.time() - t0, **test_metrics(m)}
            save_pkl({k: v for k, v in runs.items() if k[0] == ds}, res_file(ds))
        save_cache_now()
        show = [n for n in ["RL-DOHO", "DO", "HO", "GA", "HO+SI", "RL-DOHO [PERTURB only]"] if (ds, n, s) in runs]
        print(f"[{time.strftime('%H:%M:%S')}] {ds} seed {s}: " + "  ".join(f"{n}={runs[(ds, n, s)]['fit']:.4f}" for n in show)
              + f"   elapsed {time.time() - t_start:.0f}s", flush=True)
save_pkl(NEW_EVAL, f"{OUT_DIR}/v4_eval_cost_{'_'.join(DATASETS)}.pkl")
print("Runs complete for", DATASETS)

0 runs already saved
[16:17:44] WDBC seed 0: RL-DOHO=0.0266  DO=0.0361  HO=0.0211  GA=0.0187  HO+SI=0.0197  RL-DOHO [PERTURB only]=0.0225   elapsed 127s
[16:19:41] WDBC seed 1: RL-DOHO=0.0252  DO=0.0332  HO=0.0182  GA=0.0151  HO+SI=0.0179  RL-DOHO [PERTURB only]=0.0174   elapsed 245s
[16:21:35] WDBC seed 2: RL-DOHO=0.0216  DO=0.0277  HO=0.0187  GA=0.0137  HO+SI=0.0130  RL-DOHO [PERTURB only]=0.0166   elapsed 359s
[16:23:48] WDBC seed 3: RL-DOHO=0.0275  DO=0.0416  HO=0.0302  GA=0.0246  HO+SI=0.0265  RL-DOHO [PERTURB only]=0.0265   elapsed 492s
[16:25:51] WDBC seed 4: RL-DOHO=0.0231  DO=0.0351  HO=0.0226  GA=0.0204  HO+SI=0.0236  RL-DOHO [PERTURB only]=0.0176   elapsed 614s
[16:28:02] WDBC seed 5: RL-DOHO=0.0245  DO=0.0342  HO=0.0251  GA=0.0232  HO+SI=0.0218  RL-DOHO [PERTURB only]=0.0225   elapsed 746s
[16:30:11] WDBC seed 6: RL-DOHO=0.0249  DO=0.0286  HO=0.0267  GA=0.0196  HO+SI=0.0240  RL-DOHO [PERTURB only]=0.0229   elapsed 875s
[16:32:18] WDBC seed 7: RL-DOHO=0.0197  DO=0.0319  HO=0

## 7. Results (reads every dataset's saved file, so it also works after parallel sessions)

In [8]:
pd.set_option("display.width", 220)
cols = ["fit", "cv", "n_feats", "test_acc", "test_prec", "test_rec", "test_f1"]
runs = load_runs()
R = pd.DataFrame([{"dataset": k[0], "method": k[1], "seed": k[2], **{c: v[c] for c in cols}} for k, v in runs.items()])
R = R[R.seed.isin(SEEDS)]
DSS = [d for d in TAB if d in set(R.dataset)]
print(R.groupby(["dataset", "method"]).size().unstack(0).reindex(ALL).to_string())
R.to_csv(f"{OUT_DIR}/v4_all_runs.csv", index=False)
S = summary_table(R, ALL)
for ds in DSS:
    T = S[S.dataset == ds].set_index("method").reindex(ALL)
    print(f"\n{'=' * 22} {ds} {'=' * 22}")
    print(T[["fit", "fit_sd", "cv", "test_acc", "acc_sd", "test_f1", "n_feats"]].round(4).to_string())

dataset                 Backache  Dermatology  Hepatitis  SPECTF  WDBC
method                                                                
RL-DOHO                       10           10         10      10    10
DO                            10           10         10      10    10
HO                            10           10         10      10    10
DOHO                          10           10         10      10    10
GA                            10           10         10      10    10
BPSO                          10           10         10      10    10
BGWO                          10           10         10      10    10
GA+SI                         10           10         10      10    10
HO+SI                         10           10         10      10    10
RL-DOHO [PERTURB only]        10           10         10      10    10
LASSO                         10           10         10      10    10
mRMR                          10           10         10      10    10
Relief

## 8. Statistics: per-dataset Friedman + Holm-corrected Wilcoxon, pooled sign tests, ranks

In [9]:
RM = R[R.method.isin(METHODS)]
W = pd.concat([per_dataset_tests(RM, "RL-DOHO", METHODS, m, hb) for m, hb in [("fit", False), ("test_acc", True), ("test_f1", True)]])
POOL = pd.concat([pooled_sign(RM, "RL-DOHO", METHODS[1:], m, hb) for m, hb in [("fit", False), ("test_acc", True), ("test_f1", True)]])
print("=== Pooled over all datasets and seeds: RL-DOHO W/T/L, two-sided sign test, Holm-corrected per metric ===")
print(POOL.round(4).to_string(index=False))
print("\nPer-dataset Holm-corrected Wilcoxon, fitness (significant only):")
print(W[(W.metric == "fit") & (W.holm_p < 0.05)][["dataset", "vs", "W", "T", "L", "holm_p"]].round(4).to_string(index=False))
print("\nPer-dataset Friedman p (test score):", W[W.metric == "test_acc"].groupby("dataset").friedman_p.first().round(3).to_dict())
rk = mean_ranks(RM, METHODS, "fit", False); print("\nMean per-run rank on fitness:"); print(rk.assign(ALL=rk.mean(1)).round(2).to_string())
fr = mean_ranks(RM, FAMILY, "fit", False); print("\nWithin the DO/HO family:"); print(fr.assign(ALL=fr.mean(1)).round(2).to_string())
W.to_csv(f"{OUT_DIR}/v4_stats_per_dataset.csv", index=False); POOL.to_csv(f"{OUT_DIR}/v4_stats_pooled.csv", index=False)

=== Pooled over all datasets and seeds: RL-DOHO W/T/L, two-sided sign test, Holm-corrected per metric ===
  metric           vs  W  T  L  n  sign_p  holm_p
     fit           DO 50  0  0 50  0.0000  0.0000
     fit           HO 32  0 18 50  0.0649  0.1947
     fit         DOHO 40  2  8 50  0.0000  0.0000
     fit           GA  5  2 43 50  0.0000  0.0000
     fit         BPSO 20  3 27 50  0.3817  0.7634
     fit         BGWO 21  2 27 50  0.4709  0.7634
     fit        LASSO 41  0  9 50  0.0000  0.0000
     fit         mRMR 48  1  1 50  0.0000  0.0000
     fit      ReliefF 47  1  2 50  0.0000  0.0000
     fit All features 50  0  0 50  0.0000  0.0000
test_acc           DO 20  9 21 50  1.0000  1.0000
test_acc           HO 16  9 25 50  0.2110  1.0000
test_acc         DOHO 14 10 26 50  0.0807  0.7262
test_acc           GA 19  9 22 50  0.7552  1.0000
test_acc         BPSO 16 13 21 50  0.5114  1.0000
test_acc         BGWO 16  8 26 50  0.1641  1.0000
test_acc        LASSO 19  4 27 50  0.3020  1

## 9. Hypothesis H9: do the Notebook 8 findings replicate with SVM?

In [10]:
h = [sign_line(R, "RL-DOHO", "DO", "H9a RL-DOHO vs DO"), sign_line(R, "RL-DOHO", "HO", "H9b RL-DOHO vs HO"),
     sign_line(R, "GA", "RL-DOHO", "H9c GA vs RL-DOHO"), sign_line(R, "HO+SI", "HO", "H9d HO+SI vs HO"),
     sign_line(R, "GA+SI", "GA", "H9d GA+SI vs GA"), sign_line(R, "RL-DOHO", "RL-DOHO [PERTURB only]", "H9e RL-DOHO vs PERTURB only")]
print("\nTest score (reported, no hypothesis):")
for a, b in [("RL-DOHO", "DO"), ("RL-DOHO", "HO"), ("GA", "RL-DOHO"), ("HO+SI", "HO"), ("GA+SI", "GA")]:
    sign_line(R, a, b, f"   {a} vs {b}", "test_acc", True)

H9a RL-DOHO vs DO: RL-DOHO W/T/L = 50/0/0 over 50 paired runs (mean diff -0.04610), sign test p = 0.0000
H9b RL-DOHO vs HO: RL-DOHO W/T/L = 32/0/18 over 50 paired runs (mean diff -0.00371), sign test p = 0.0649
H9c GA vs RL-DOHO: GA W/T/L = 43/2/5 over 50 paired runs (mean diff -0.01547), sign test p = 0.0000
H9d HO+SI vs HO: HO+SI W/T/L = 37/5/8 over 50 paired runs (mean diff -0.01007), sign test p = 0.0000
H9d GA+SI vs GA: GA+SI W/T/L = 16/17/17 over 50 paired runs (mean diff +0.00098), sign test p = 1.0000
H9e RL-DOHO vs PERTURB only: RL-DOHO W/T/L = 8/1/41 over 50 paired runs (mean diff +0.00572), sign test p = 0.0000

Test score (reported, no hypothesis):
   RL-DOHO vs DO: RL-DOHO W/T/L = 20/9/21 over 50 paired runs (mean diff +0.00314), sign test p = 1.0000
   RL-DOHO vs HO: RL-DOHO W/T/L = 16/9/25 over 50 paired runs (mean diff -0.01667), sign test p = 0.2110
   GA vs RL-DOHO: GA W/T/L = 22/9/19 over 50 paired runs (mean diff +0.00579), sign test p = 0.7552
   HO+SI vs HO: HO+SI

## 10. SVM against KNN: same methods, same splits (Notebook 8 results needed in `KNN_DIR`)

In [11]:
kf = f"{KNN_DIR}/v3_all_runs.csv"
if os.path.exists(kf):
    K = pd.read_csv(kf); K = K[K.dataset.isin(DSS) & K.method.isin(ALL) & K.seed.isin(SEEDS)]
    both = R.merge(K, on=["dataset", "method", "seed"], suffixes=("_svm", "_knn"))
    print("Mean test balanced accuracy, SVM vs KNN:")
    print(both.groupby(["dataset"])[["test_acc_svm", "test_acc_knn"]].mean().round(4).to_string())
    print("\nMean per-run fitness rank of each method under each classifier:")
    rs = mean_ranks(R[R.method.isin(METHODS)], METHODS, "fit", False).mean(1)
    rk = mean_ranks(K[K.method.isin(METHODS)], METHODS, "fit", False).mean(1)
    print(pd.DataFrame({"SVM": rs, "KNN": rk}).round(2).to_string())
    from scipy.stats import spearmanr
    print(f"\nSpearman correlation of the two method rankings: {spearmanr(rs, rk).correlation:.2f}")
    print("\nPooled W/T/L under each classifier:")
    for a, b in [("RL-DOHO", "DO"), ("RL-DOHO", "HO"), ("GA", "RL-DOHO"), ("HO+SI", "HO"), ("GA+SI", "GA"), ("RL-DOHO", "RL-DOHO [PERTURB only]")]:
        sign_line(R, a, b, f"  SVM  {a} vs {b}"); sign_line(K, a, b, f"  KNN  {a} vs {b}")
else:
    print("Notebook 8 results not found at", kf)

Mean test balanced accuracy, SVM vs KNN:
             test_acc_svm  test_acc_knn
dataset                                
Backache           0.5187        0.5365
Dermatology        0.9613        0.9537
Hepatitis          0.6366        0.6366
SPECTF             0.7635        0.7323
WDBC               0.9511        0.9453

Mean per-run fitness rank of each method under each classifier:
                SVM    KNN
method                    
RL-DOHO        3.82   4.15
DO             8.98   8.80
HO             4.40   4.75
DOHO           6.13   5.55
GA             1.58   1.82
BPSO           3.18   3.28
BGWO           3.53   3.29
LASSO          6.26   6.80
mRMR           8.81   8.22
ReliefF        8.52   8.54
All features  10.79  10.80

Spearman correlation of the two method rankings: 0.99

Pooled W/T/L under each classifier:
  SVM  RL-DOHO vs DO: RL-DOHO W/T/L = 50/0/0 over 50 paired runs (mean diff -0.04610), sign test p = 0.0000
  KNN  RL-DOHO vs DO: RL-DOHO W/T/L = 49/0/1 over 50 paired run

## 11. Summary
Per-run results: `v4_runs_<dataset>.pkl` and `v4_all_runs.csv`. Statistics: `v4_stats_*.csv`. All files are in `OUT_DIR`.